# BraTS Segmentation Visualization

Visualizing best model predictions vs ground truth on validation cases.

In [0]:
%pip install -q "monai[nibabel,tqdm,einops]>=1.4" einops "nibabel>=5.2" "connected-components-3d>=3.12" "scipy>=1.11" "scikit-learn>=1.4" "matplotlib>=3.8" "pyyaml>=6.0" "openpyxl>=3.1" /Workspace/Users/soumyanil.banerjee@corewellhealth.org/Brain_Tumor_Segmentation

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from brats.models.multitask import load_model_from_checkpoint, count_parameters
from brats.data.preprocess import dequantize, labels_to_regions
from brats.data.transforms import load_records
from brats.config import DataConfig
from brats.constants import REGIONS, SEQUENCES, COHORTS
from monai.inferers import sliding_window_inference

# Set data root
os.environ["BRATS_CACHE_ROOT"] = "/Volumes/imaging_research/brain_tumor_segmentation/brats_dataset"

# Load best checkpoint
CKPT_PATH = "/Workspace/Users/soumyanil.banerjee@corewellhealth.org/Brain_Tumor_Segmentation/checkpoints/segresnet_base/best_avg.pt"
model = load_model_from_checkpoint(CKPT_PATH, map_location="cuda:0")
model = model.cuda().eval()

total, trainable = count_parameters(model)
ckpt = torch.load(CKPT_PATH, map_location="cuda:0", weights_only=False)
print(f"Model: {total/1e6:.1f}M params")
print(f"Checkpoint epoch: {ckpt.get('epoch', '?')}")
for key in ["avg", "ET", "TC", "WT"]:
    best_key = f"best_{key}"
    if best_key in ckpt:
        print(f"  Best dice {key}: {ckpt[best_key]:.4f}")

In [0]:
import pandas as pd

CACHE_ROOT = "/Volumes/imaging_research/brain_tumor_segmentation/brats_dataset"
SPLITS_CSV = "/Workspace/Users/soumyanil.banerjee@corewellhealth.org/Brain_Tumor_Segmentation/splits/split_random_seed42.csv"

# Load val records directly from split CSV
df = pd.read_csv(SPLITS_CSV)
val_df = df[df["split"] == "val"]
print(f"Total val records: {len(val_df)}")

# Pick cases with the LARGEST tumors (most visible) per cohort
candidates = {c: [] for c in COHORTS}
for _, row in val_df.iterrows():
    cohort = row["cohort"]
    npz_path = f"{CACHE_ROOT}/{cohort}/{row['case_id']}.npz"
    with np.load(npz_path) as z:
        if "seg" in z.files:
            wt_vol = (z["seg"] > 0).sum()  # WT = any nonzero label
            et_vol = (z["seg"] == 3).sum()  # ET = label 3
            candidates[cohort].append({
                "case_id": row["case_id"], "cohort": cohort,
                "cache_path": npz_path, "wt_vol": int(wt_vol), "et_vol": int(et_vol),
            })

# Pick the top 2 largest-tumor cases per cohort (6 total)
selected = {}
selected_list = []
for cohort in COHORTS:
    if candidates[cohort]:
        top2 = sorted(candidates[cohort], key=lambda x: x["wt_vol"], reverse=True)[:2]
        for i, case in enumerate(top2):
            key = f"{cohort}_{i}"
            selected[key] = case
            selected_list.append(case)
            print(f"  {cohort} #{i+1}: {case['case_id']} (WT={case['wt_vol']:,} voxels, ET={case['et_vol']:,} voxels)")

case_labels = [v['cohort'] + '/' + v['case_id'] for v in selected_list]
print(f"Selected cases ({len(selected_list)}): {case_labels}")

# Run inference
results = []
for rec in selected_list:
    with np.load(rec["cache_path"]) as z:
        img = dequantize(z["img"]).astype(np.float32)   # (4, D, H, W)
        seg = labels_to_regions(z["seg"]).astype(np.float32)  # (3, D, H, W)

    # Prepare input tensor
    x = torch.from_numpy(img).unsqueeze(0).cuda()  # (1, 4, D, H, W)

    # Sliding window inference
    def model_fn(patch):
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            out = model(patch, gt_wt=None)
        return out["seg"]

    with torch.no_grad():
        logits = sliding_window_inference(
            x, roi_size=(128, 128, 128), sw_batch_size=4,
            predictor=model_fn, overlap=0.5, mode="gaussian",
        )

    pred = (torch.sigmoid(logits).cpu().numpy()[0] > 0.5).astype(np.float32)  # (3, D, H, W)

    # Compute dice per region
    dice_scores = {}
    for i, region in enumerate(REGIONS):
        p, g = pred[i], seg[i]
        intersection = (p * g).sum()
        denom = p.sum() + g.sum()
        dice_scores[region] = (2.0 * intersection / denom).item() if denom > 0 else 1.0

    results.append({
        "case_id": rec["case_id"],
        "cohort": cohort,
        "img": img,
        "seg_gt": seg,
        "seg_pred": pred,
        "dice": dice_scores,
    })
    print(f"  {rec['cohort']}/{rec['case_id']}: Dice ET={dice_scores['ET']:.3f} TC={dice_scores['TC']:.3f} WT={dice_scores['WT']:.3f}")

print(f"\nInference complete on {len(results)} cases.")

In [0]:
def find_tumor_center(wt_mask):
    """Find the center of the WT mask for optimal slice selection."""
    coords = np.argwhere(wt_mask > 0)
    if len(coords) == 0:
        return tuple(s // 2 for s in wt_mask.shape)
    return tuple(coords.mean(axis=0).astype(int))

def overlay_seg(img_slice, seg_slice, alpha=0.55):
    """Overlay 3-channel segmentation on grayscale MRI slice.
    Paint in reverse nesting order (WT first, TC on top, ET on top of that)
    so the smallest region is always visible.
    Colors: WT=blue, TC=yellow, ET=bright red."""
    vmin, vmax = np.percentile(img_slice[img_slice > 0], [1, 99]) if (img_slice > 0).any() else (0, 1)
    normed = np.clip((img_slice - vmin) / (vmax - vmin + 1e-8), 0, 1)
    rgb = np.stack([normed, normed, normed], axis=-1)

    # Paint WT (outermost) first, TC next, ET (innermost) last on top
    region_colors = {
        2: np.array([0.2, 0.4, 1.0]),   # WT = blue
        1: np.array([1.0, 0.9, 0.0]),   # TC = yellow
        0: np.array([1.0, 0.1, 0.1]),   # ET = bright red
    }
    for idx in [2, 1, 0]:
        mask = seg_slice[idx] > 0
        if mask.any():
            for c in range(3):
                rgb[..., c] = np.where(mask, rgb[..., c] * (1 - alpha) + region_colors[idx][c] * alpha, rgb[..., c])
    return np.clip(rgb, 0, 1)

def visualize_case(result, figsize=(18, 16)):
    """3x3 grid: rows = MRI / GT overlay / Pred overlay.
    Columns = Sagittal / Coronal / Axial.
    BraTS data in LPS: dim 0 = L-R (sagittal), dim 1 = P-A (coronal), dim 2 = S-I (axial)."""
    img = result["img"]       # (4, D, H, W)
    gt = result["seg_gt"]     # (3, D, H, W)
    pred = result["seg_pred"] # (3, D, H, W)
    dice = result["dice"]
    t1c = img[1]  # T1c channel for display

    # Find tumor center from GT WT mask
    center = find_tumor_center(gt[2])  # WT is index 2
    d, h, w = center

    # BraTS LPS orientation:
    #   Dim 0 (d) slicing -> Sagittal view (L-R axis)
    #   Dim 1 (h) slicing -> Coronal view (P-A axis)
    #   Dim 2 (w) slicing -> Axial view (S-I axis)
    slices_img  = [t1c[d, :, :],      t1c[:, h, :],      t1c[:, :, w]]
    slices_gt   = [gt[:, d, :, :],     gt[:, :, h, :],    gt[:, :, :, w]]
    slices_pred = [pred[:, d, :, :],   pred[:, :, h, :],  pred[:, :, :, w]]
    titles_col  = [f"Sagittal (slice {d})", f"Coronal (slice {h})", f"Axial (slice {w})"]

    fig, axes = plt.subplots(3, 3, figsize=figsize)
    fig.suptitle(
        f"{result['case_id']} ({result['cohort']})  |  "
        f"Dice: ET={dice['ET']:.3f}  TC={dice['TC']:.3f}  WT={dice['WT']:.3f}",
        fontsize=15, fontweight="bold", y=0.98
    )

    row_configs = [
        ("T1c MRI",           None,          None),
        ("GROUND TRUTH",      slices_gt,     "#006400"),
        ("MODEL PREDICTION",  slices_pred,   "#8B0000"),
    ]

    for col in range(3):
        for row in range(3):
            ax = axes[row, col]
            s_img = slices_img[col]

            if row == 0:
                # MRI only
                vmin_v, vmax_v = np.percentile(s_img[s_img > 0], [1, 99]) if (s_img > 0).any() else (0, 1)
                ax.imshow(s_img.T, cmap="gray", origin="lower", vmin=vmin_v, vmax=vmax_v)
            else:
                # Overlay
                seg_data = row_configs[row][1][col]
                ax.imshow(overlay_seg(s_img, seg_data).transpose(1, 0, 2), origin="lower")

            ax.set_xticks([]); ax.set_yticks([])

            # Column titles on top row
            if row == 0:
                ax.set_title(titles_col[col], fontsize=14, fontweight="bold", pad=8)

            # Row labels on left side as large visible text
            if col == 0:
                label_text = row_configs[row][0]
                label_color = row_configs[row][2] or "white"
                ax.text(-0.02, 0.5, label_text, transform=ax.transAxes,
                        fontsize=13, fontweight="bold", color=label_color,
                        va="center", ha="right", rotation=90,
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.7) if row > 0 else None)

    # Legend with matching colors
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#CC1A1A", label="ET (Enhancing Tumor)"),
        Patch(facecolor="#E6E600", edgecolor="gray", label="TC (Tumor Core)"),
        Patch(facecolor="#3366FF", label="WT (Whole Tumor)"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=12,
              bbox_to_anchor=(0.5, 0.005), frameon=True, edgecolor="gray", fancybox=True)
    plt.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05, hspace=0.08, wspace=0.05)
    plt.show()

# Visualize all selected cases
for r in results:
    visualize_case(r)

In [0]:
import mlflow
import tempfile

EXP_NAME = "/Users/soumyanil.banerjee@corewellhealth.org/brats_segresnet_training"
TRAIN_RUN_ID = "c611aaa1309f413b828cf3995341bdc5"

mlflow.set_experiment(EXP_NAME)
client = mlflow.tracking.MlflowClient()

# --- 1. Log eval dice metrics to the training run ---
print("Logging eval metrics to training run...")
for r in results:
    case_tag = r["case_id"].replace("-", "_")
    for region, score in r["dice"].items():
        metric_key = f"eval/{r['cohort']}/{region}"
        client.log_metric(TRAIN_RUN_ID, metric_key, score)

# Log per-cohort averages
for cohort in COHORTS:
    cohort_results = [r for r in results if r["cohort"] == cohort]
    if cohort_results:
        for region in REGIONS:
            avg = np.mean([r["dice"][region] for r in cohort_results])
            client.log_metric(TRAIN_RUN_ID, f"eval/{cohort}/avg_{region}", avg)
        overall = np.mean([r["dice"][reg] for r in cohort_results for reg in REGIONS])
        client.log_metric(TRAIN_RUN_ID, f"eval/{cohort}/avg_all", overall)

# Log global averages
all_dice = {region: np.mean([r["dice"][region] for r in results]) for region in REGIONS}
for region, val in all_dice.items():
    client.log_metric(TRAIN_RUN_ID, f"eval/mean_{region}", val)
client.log_metric(TRAIN_RUN_ID, "eval/mean_dice", np.mean(list(all_dice.values())))
print(f"  Mean dice: ET={all_dice['ET']:.3f}  TC={all_dice['TC']:.3f}  WT={all_dice['WT']:.3f}")
print(f"  Overall mean: {np.mean(list(all_dice.values())):.3f}")

# --- 2. Save visualization plots as PNG and log as artifacts ---
with tempfile.TemporaryDirectory() as tmpdir:
    print(f"\nSaving plots to temp dir and logging as artifacts...")

    for idx, r in enumerate(results):
        fig = plt.figure(figsize=(18, 16))
        fig.suptitle(
            f"{r['case_id']} ({r['cohort']})  |  "
            f"Dice: ET={r['dice']['ET']:.3f}  TC={r['dice']['TC']:.3f}  WT={r['dice']['WT']:.3f}",
            fontsize=15, fontweight="bold", y=0.98
        )
        axes = fig.subplots(3, 3)
        t1c = r["img"][1]
        center = find_tumor_center(r["seg_gt"][2])
        d, h, w = center

        slices_img  = [t1c[d, :, :], t1c[:, h, :], t1c[:, :, w]]
        slices_gt   = [r["seg_gt"][:, d, :, :], r["seg_gt"][:, :, h, :], r["seg_gt"][:, :, :, w]]
        slices_pred = [r["seg_pred"][:, d, :, :], r["seg_pred"][:, :, h, :], r["seg_pred"][:, :, :, w]]
        titles_col  = [f"Sagittal (slice {d})", f"Coronal (slice {h})", f"Axial (slice {w})"]
        row_configs = [
            ("T1c MRI",           None,           None),
            ("GROUND TRUTH",      slices_gt,      "#006400"),
            ("MODEL PREDICTION",  slices_pred,    "#8B0000"),
        ]

        for col in range(3):
            for row in range(3):
                ax = axes[row, col]
                s_img = slices_img[col]
                if row == 0:
                    vmin_v, vmax_v = np.percentile(s_img[s_img > 0], [1, 99]) if (s_img > 0).any() else (0, 1)
                    ax.imshow(s_img.T, cmap="gray", origin="lower", vmin=vmin_v, vmax=vmax_v)
                else:
                    seg_data = row_configs[row][1][col]
                    ax.imshow(overlay_seg(s_img, seg_data).transpose(1, 0, 2), origin="lower")
                ax.set_xticks([]); ax.set_yticks([])
                if row == 0:
                    ax.set_title(titles_col[col], fontsize=14, fontweight="bold", pad=8)
                if col == 0:
                    label_text = row_configs[row][0]
                    label_color = row_configs[row][2] or "white"
                    ax.text(-0.02, 0.5, label_text, transform=ax.transAxes,
                            fontsize=13, fontweight="bold", color=label_color,
                            va="center", ha="right", rotation=90,
                            bbox=dict(boxstyle="round,pad=0.3", facecolor="black", alpha=0.7) if row > 0 else None)

        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor="#CC1A1A", label="ET (Enhancing Tumor)"),
            Patch(facecolor="#E6E600", edgecolor="gray", label="TC (Tumor Core)"),
            Patch(facecolor="#3366FF", label="WT (Whole Tumor)"),
        ]
        fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=12,
                   bbox_to_anchor=(0.5, 0.005), frameon=True, edgecolor="gray", fancybox=True)
        plt.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05, hspace=0.08, wspace=0.05)

        fname = f"{r['cohort']}_{r['case_id']}.png"
        fpath = os.path.join(tmpdir, fname)
        fig.savefig(fpath, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Saved {fname}")

    # Log all PNGs as artifacts under "eval_visualizations" folder
    client.log_artifacts(TRAIN_RUN_ID, tmpdir, artifact_path="eval_visualizations")
    print(f"\nLogged {len(results)} plots to MLflow run {TRAIN_RUN_ID[:12]}... under 'eval_visualizations/'")

print("Done!")